# **PPO and GPT2**

1. LLM: FLAN-T5.
2. Reward model: of Meta AI's, a binary classifier that predicts whether a given text is "not hate" or "hate."
3. Reinforcement Learning algorithm: Proximal Policy Optimization (PPO)
2. Objective: Generate less toxic content.

In [ ]:
!pip install --upgrade pip
!pip install --disable-pip-version-check \
    torch==1.13.1 \
    torchdata==0.5.1

!pip install \
    transformers==4.27.2 \
    datasets==2.11.0 \
    evaluate==0.4.0 \
    rouge_score==0.1.2 \
    peft==0.3.0 --quiet \
    trl==0.4.4 \
    loralib==0.1.1

# Installing the Reinforcement Learning library directly from github.
!pip install git+https://github.com/lvwerra/trl.git@25fa1bd

In [ ]:
from datasets import  Dataset
from transformers import pipeline, AutoModelForSequenceClassification, AutoModelForSeq2SeqLM, AutoTokenizer,GenerationConfig, TrainingArguments,Trainer

from trl import PPOTrainer, PPOConfig, AutoModelForSeq2SeqLMWithValueHead
from trl import create_reference_model
from trl.core import LengthSampler

import pandas as pd
import torch
import evaluate
import numpy as np

from tqdm import tqdm
tqdm.pandas()

from peft import LoraConfig, TaskType, PeftModel

### **I. Dataset**

In [ ]:
# Load the datasets
train_path = r"/kaggle/input/training-dataset-peft-lora/dataset_peft_lora/knkarthick_dialogsum/train.csv"
test_path = r"/kaggle/input/training-dataset-peft-lora/dataset_peft_lora/knkarthick_dialogsum/validation.csv"

dataset_train = pd.read_csv(train_path)
dataset_test = pd.read_csv(test_path)

dataset_train.head(3)
dataset_test.head(4)

,id,dialogue,summary,topic
0,test_0_1,"#Person1#: Ms. Dawson, I need you to take a di...",Ms. Dawson helps #Person1# to write a memo to ...,communication method
1,test_0_2,"#Person1#: Ms. Dawson, I need you to take a di...",In order to prevent employees from wasting tim...,company policy
2,test_0_3,"#Person1#: Ms. Dawson, I need you to take a di...",Ms. Dawson takes a dictation for #Person1# abo...,dictation
3,test_1_1,#Person1#: You're finally here! What took so l...,#Person2# arrives late because of traffic jam....,public transportation


In [ ]:
#Pre-Process the dataset

def build_dataset(model_name,
                  dataset,
                  input_min_text_length,
                  input_max_text_length):


    dataset = Dataset.from_pandas(dataset)
    dataset = dataset.filter(lambda x:len(x['dialogue'])> input_min_text_length and len(x['dialogue']) <= input_max_text_length)

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # Create instruction-based prompts to model summarization.
    def tokenize(sample):
        prompt=f"""Summarize the following conversation.
                   {sample['dialogue']}

                   Summary:
                """
        sample["input_ids"] = tokenizer.encode(prompt)
        sample["query"] = tokenizer.decode(sample["input_ids"])
        return sample

    # tokenize each dialogue
    dataset = dataset.map(tokenize)
    dataset.set_format(type="torch")

    #split the dataset into train and test parts
    dataset_splits = dataset.train_test_split(test_size=0.2, shuffle=False, seed=42)

    return dataset_splits


model_name = 'google/flan-t5-base'

dataset = build_dataset(model_name = model_name,
                        dataset = dataset_train,
                        input_min_text_length=200,
                        input_max_text_length=1000)

Filter:   0%|          | 0/12460 [00:00<?, ? examples/s]

Map:   0%|          | 0/10022 [00:00<?, ? examples/s]

In [ ]:

lora_config = LoraConfig(
    r=32,
    lora_alpha=32,
    target_modules=["q","v"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM #FLAN-T%
)

In [ ]:
peft_model_base= AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base", torch_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")

peft_model = PeftModel.from_pretrained(peft_model_base,
                                       '/kaggle/input/peft-dialogue-summary-checkpoint-local/transformers/default/1/peft-dialogue-summary-checkpoint-local/',
                                       torch_dtype=torch.bfloat16,
                                       is_trainable=False,
                                      )

# **II. Create a ppo model from peft_model**

In [ ]:
#create a ppo model from peft_model
ppo_model = AutoModelForSeq2SeqLMWithValueHead.from_pretrained(peft_model,
                                                               torch_dtype=torch.bfloat16,
                                                               is_trainable=True)

print(ppo_model.v_head)

PPO model prameters to be updated(ValueHead+769 params):

trainable model

 parameters:769

 all model parameters 251117569

 percentrage of trainable model: 0.0003062310626302694

ValueHead(

  (dropout): Dropout(p=0.1, inplace=False)

  (summary): Linear(in_features=768, out_features=1, bias=True)

  (flatten): Flatten(start_dim=1, end_dim=-1)

)


In [ ]:
#Reference Model:a model that is not fine tuned with all, not even with PEFT.
ref_model = create_reference_model(ppo_model)

### **3. Reaward model**

In [ ]:
toxicity_model_name = "facebook/roberta-hate-speech-dynabench-r4-target"
toxicity_tokenizer = AutoTokenizer.from_pretrained(toxicity_model_name, device_map="auto")
toxicity_model = AutoModelForSequenceClassification.from_pretrained(toxicity_model_name, device_map="auto")

print(toxicity_model.config.id2label)

tokenizer_config.json:   0%|          | 0.00/1.11k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/816 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

{0: 'nothate', 1: 'hate'}


In [ ]:
non_toxic_text = " i want to kiss you"

toxicity_input_ids = toxicity_tokenizer(non_toxic_text, return_tensors="pt").input_ids.to(device) #added

logits = toxicity_model(input_ids= toxicity_input_ids).logits
print(f"logits[not hate, hate:{logits.tolist()[0]}")

probabilities = logits.softmax(dim=1).tolist()[0]
print(f"probabilities [not hate, hate]:{probabilities}")

not_hate_index = 0
nothate_reward=(logits[:,not_hate_index]).tolist()
print(f"reward (high):{nothate_reward}")

logits[not hate, hate:[1.6786898374557495, -1.5461454391479492]

probabilities [not hate, hate]:[0.9617581963539124, 0.03824174404144287]

reward (high):[1.6786898374557495]


In [ ]:
toxic_text = "you are disgusting and terrible and i damm hate you "

toxicity_input_ids = toxicity_tokenizer(non_toxic_text, return_tensors="pt").input_ids.to(device) #added

logits = toxicity_model(input_ids= toxicity_input_ids).logits
print(f"logits[not hate, hate:{logits.tolist()[0]}")

#print the probabilities for [not hate, hate]
probabilities = logits.softmax(dim=1).tolist()[0]
print(f"probabilities [not hate, hate]:{probabilities}")

#get the togits for "not hate" - this is reward
not_hate_index = 0
notehate_reward=(logits[:,not_hate_index]).tolist()
print(f"reward (high):{nothate_reward}")


logits[not hate, hate:[1.6786898374557495, -1.5461454391479492]

probabilities [not hate, hate]:[0.9617581963539124, 0.03824174404144287]

reward (high):[1.6786898374557495]


### **4. Evaluate Toxicity of Reward Model**

In [ ]:

device = "cpu"

sentiment_pipe = pipeline("sentiment-analysis",
                          model = toxicity_model_name,
                          framework="pt",
                          device = device)

reward_logits_kwargs = {
    "top_k":None,
    "function_to_apply":"none",
    "batch_size":16
}

reward_probabilities_kwards = {
    "top_k":None,
    "function_to_apply":"softmax",
    "batch_size":16
}

print("Reward model output for non-toxic text:")
print(sentiment_pipe(non_toxic_text,**reward_logits_kwargs))
print(sentiment_pipe(non_toxic_text,**reward_probabilities_kwards))

print("\nReward model output for toxic text:")
print(sentiment_pipe(toxic_text, **reward_logits_kwargs))
print(sentiment_pipe(toxic_text,**reward_probabilities_kwards))

Reward model output for non-toxic text:

[{'label': 'nothate', 'score': 1.6786898374557495}, {'label': 'hate', 'score': -1.5461454391479492}]

[{'label': 'nothate', 'score': 0.9617581963539124}, {'label': 'hate', 'score': 0.03824174404144287}]



 Reward model output for toxic text:

[{'label': 'hate', 'score': 2.475832462310791}, {'label': 'nothate', 'score': -2.809473991394043}]

[{'label': 'hate', 'score': 0.9949600696563721}, {'label': 'nothate', 'score': 0.005039950367063284}]


In [ ]:
toxicity_evaluator = evaluate.load("toxicity", toxicity_model_name,
                                   module_type="measurement",
                                   toxic_label="hate",
                                  )

In [ ]:
toxicity_score = toxicity_evaluator.compute(predictions=[
    non_toxic_text
])
print("Toxicity score for non-toxic text:")
print(toxicity_score["toxicity"])

toxicity_socre = toxicity_evaluator.compute(predictions=[
    toxic_text
])

print("\n Toxicity socre for toxic text:")
print(toxicity_score["toxicity"])

Toxicity score for non-toxic text:

[0.03824174404144287]



 Toxicity socre for toxic text:

[0.03824174404144287]


In [ ]:
#function to evaluate and pass the dataset
def evaluate_toxicity(model,
                      toxicity_evaluator,
                      tokenizer,
                      dataset,
                      num_samples):
    max_new_tokens=100

    toxicities = []
    for i, sample in tqdm(enumerate(dataset)):
        input_text = sample["query"]
        if i > num_samples:
            break
        input_ids = tokenizer(input_text, return_tensors = "pt", padding=True).input_ids

        generation_config = GenerationConfig(max_new_tokens = max_new_tokens,
                                             tok_k=0.0,
                                             top_p=1.0,
                                             do_sample=True)
        response_token_ids = model.generate(input_ids = input_ids, generation_config = generation_config)
        generated_text = tokenizer.decode(response_token_ids[0], skip_special_tokens=True)

        toxicity_score = toxicity_evaluator.compute(predictions=[(input_text+""+generated_text)])

        toxicities.extend(toxicity_score["toxicity"])

    #compute mean & std
    mean = np.mean(toxicities)
    std = np.std(toxicities)

    return mean, std

# **II. Evaluate Toxicity with PEFT_fine_tune_model/BEFORE DETOX**

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name, device_map="auto") #'google/flan-t5-base'
mean_before_detoxification, std_before_detoxicifcation = evaluate_toxicity(model = ref_model, #create_reference_model(ppo_model/peft_fine_tune_model)
                                                                           toxicity_evaluator = toxicity_evaluator,
                                                                           tokenizer = tokenizer, #'google/flan-t5-base'
                                                                           dataset= dataset["test"],
                                                                           num_samples=10)

print(f'toxicity [mean,std] before detox:{mean_before_detoxification,std_before_detoxicifcation}')

11it [00:17,  1.59s/it]

toxicity [mean,std] before detox:(0.010048318749547681, 0.014725097147429488)


1. Perfomr Fine-Tuning to Detoxify the summaries

In [ ]:
# Initialize the PPOTrainer
# Load the ppo_model and tokenizer, along with a frozen version of the model called ref_model.
# The first model will be optimized, while the second model is used as a reference to calculate the KL-divergence
# from the starting point. This serves as an additional reward signal during PPO training, ensuring the optimized model
# does not deviate significantly from the original LLM.

learning_rate = 1.41e-5
max_ppo_epochs=1
mini_batch_size=4
batch_size=16

config = PPOConfig(
    model_name = model_name,
    learning_rate = learning_rate,
    ppo_epochs=max_ppo_epochs,
    mini_batch_size=mini_batch_size,
    batch_size=batch_size
)

def collator(data):
    return dict((key,[d[key] for d in data]) for key in data[0])

ppo_trainer = PPOTrainer(config = config,
                         model = ppo_model,
                         ref_model = ref_model, #oringinal model from lab 2
                         tokenizer= tokenizer,
                         dataset =dataset['train'],
                         data_collator = collator)

2. Fine-tune the model

In [ ]:
output_min_length = 100
output_max_length = 400
output_length_sampler = LengthSampler(output_min_length,output_max_length)

generation_kwards= {
    "min_length":5,
    "top_k": 0.0,
    "top_p":1.0,
    "do_sample":True
}

reward_kwargs = {
    "top_k": None,
    "function_to_apply":"none",
    "batch_size":16
}

max_ppo_steps = 10

for step,batch in tqdm(enumerate(ppo_trainer.dataloader)):
    #break when we teach max_steps
    if step >= max_ppo_steps:
        break

    prompt_tensors = batch["input_ids"]

    #get response from FLAN-T5/PEFT LLM
    summary_tensors = []

    for prompt_tensor in prompt_tensors:
        max_new_tokens = output_length_sampler()

        generation_kwards["max_new_tokens"]=max_new_tokens
        summary = ppo_trainer.generate(prompt_tensor, **generation_kwards)

        summary_tensors.append(summary.squeeze()[-max_new_tokens:])

    #this needs to be called "response"
    batch["response"]=[tokenizer.decode(r.squeeze()) for r in summary_tensors]

    #compute reward outputs
    query_response_pairs = [q+r for q, r in zip(batch["query"],batch["response"])]
    rewards = sentiment_pipe(query_response_pairs, **reward_kwargs)
    #use the nothate item beacuse this is the score for the positive nothate class.
    reward_tensors = [torch.tensor(reward[not_hate_index]["score"]) for reward in rewards]

    #run PPO step
    stats = ppo_trainer.step(prompt_tensors, summary_tensors, reward_tensors)
    ppo_trainer.log_stats(stats, batch, reward_tensors)

    print(f"objective/kl: {stats['objective/kl']}")
    print(f'ppo/returns/mean:{stats["ppo/returns/mean"]}')
    print(f'ppo/policy/advantages_mean:{stats["ppo/policy/advantages_mean"]}')
    print('-'.join('' for x in range(100)))

0it [00:00, ?it/s]You're using a T5TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.

1it [02:05, 125.59s/it]

objective/kl: 0.04473315179347992

ppo/returns/mean:1.3326596021652222

ppo/policy/advantages_mean:-2.155499068123845e-08

---------------------------------------------------------------------------------------------------


2it [04:03, 120.82s/it]

objective/kl: 0.033818021416664124

ppo/returns/mean:1.773255467414856

ppo/policy/advantages_mean:1.2268441196283675e-07

---------------------------------------------------------------------------------------------------


3it [06:23, 129.97s/it]

...

objective/kl: -0.05551474913954735

ppo/returns/mean:1.518635630607605

ppo/policy/advantages_mean:-1.7418450681816466e-08

---------------------------------------------------------------------------------------------------


9it [20:46, 146.14s/it]

objective/kl: 0.03832945227622986

ppo/returns/mean:1.8590888977050781

ppo/policy/advantages_mean:3.924174407643477e-08

---------------------------------------------------------------------------------------------------


10it [22:54, 137.43s/it]

objective/kl: 0.004587208852171898

ppo/returns/mean:1.7363693714141846

ppo/policy/advantages_mean:4.795050756456476e-08

---------------------------------------------------------------------------------------------------


# **III. Evaluate the model quantitative, after detox**

In [ ]:
#load the PPO/PEFT model back from disk and use the test dataset split to evaluate the toxicity score of the RL-fine-tuned model

mean_after_detoxification, std_after_detoxification = evaluate_toxicity(model=ppo_model,
                                                                        toxicity_evaluator = toxicity_evaluator,
                                                                        tokenizer = tokenizer, #'google/flan-t5-base'
                                                                        dataset= dataset["test"],
                                                                        num_samples=10)

print(f'toxicity[mean,std] after detox:[{mean_after_detoxification},{std_after_detoxification}]')

11it [00:17,  1.59s/it]

toxicity[mean,std] after detox:[0.021689416068098086,0.0420182407059758]


1. Create response dataset

In [ ]:
batch_size = 20
compare_results = {}

df_batch = dataset["test"][0:batch_size]

compare_results["query"] = df_batch["query"]
prompt_tensors = df_batch["input_ids"]

summary_tensors_ref = []
summary_tensors = []

# Get response from ppo and base model.
for i in tqdm(range(batch_size)):
    gen_len = output_length_sampler()
    generation_kwards["max_new_tokens"] = gen_len

    summary = ref_model.generate(
        input_ids=torch.as_tensor(prompt_tensors[i]).unsqueeze(dim=0).to(device),
        **generation_kwards
    ).squeeze()[-gen_len:]
    summary_tensors_ref.append(summary)

    summary = ppo_model.generate(
        input_ids=torch.as_tensor(prompt_tensors[i]).unsqueeze(dim=0).to(device),
        **generation_kwards
    ).squeeze()[-gen_len:]
    summary_tensors.append(summary)

# Decode responses.
compare_results["response_before"] = [tokenizer.decode(summary_tensors_ref[i]) for i in range(batch_size)]
compare_results["response_after"] = [tokenizer.decode(summary_tensors[i]) for i in range(batch_size)]

# Sentiment analysis of query/response pairs before/after.
texts_before = [d + s for d, s in zip(compare_results["query"], compare_results["response_before"])]
rewards_before = sentiment_pipe(texts_before, **reward_kwargs)
compare_results["reward_before"] = [reward[not_hate_index]["score"] for reward in rewards_before]

texts_after = [d + s for d, s in zip(compare_results["query"], compare_results["response_after"])]
rewards_after = sentiment_pipe(texts_after, **reward_kwargs)
compare_results["reward_after"] = [reward[not_hate_index]["score"] for reward in rewards_after]

100%|██████████████████████████████████████████████████████████████████████████████████| 20/20 [01:04<00:00,  3.24s/it]


2. Display result

In [ ]:
#display the results
pd.set_option('display.max_colwidth', 500)
df_compare_results = pd.DataFrame(compare_results)
df_compare_results["reward_diff"] = df_compare_results['reward_after'] - df_compare_results['reward_before']
df_compare_results_sorted = df_compare_results.sort_values(by=['reward_diff'], ascending=False).reset_index(drop=True)
df_compare_results_sorted

,query,response_before,response_after,reward_before,reward_after,reward_diff
0,"Summarize the following conversation. #Person1#: It smells like an ashtray in here! #Person2#: Hi honey! What's wrong? Why do you have that look on your face? #Person1#: What's wrong? I thought we agreed that you were gonna quit smoking. #Person2#: No! I said I was going to cut down which is very different. You can't just expect me to go cold turkey overnight! #Person1#: Look, there are other ways to quit. You can try the nicotine patch, or nicotine chewing gum. We spend a fortune on cigaret...",<pad> Overcome your exhaustion problems.</s>,"<pad> People smoke cigarettes in public places now, but some people will still smoke now. There is no option for companies to help them out.</s>",1.313871,3.672170,2.358299
1,"Summarize the following conversation. #Person1#: How much are you asking for this? #Person2#: I'm offering them to you at 150 yuan a piece. Is that all right? #Person1#: Is tax already included in their price? #Person2#: Yes. Our price can't be matched. #Person1#: Would you consider a volume discount? #Person2#: If you buy 1, 000 or more, you'll get a 10 % discount. #Person1#: I'll accept your offer. Summary: </s>",<pad> The price is the same as the correct price on the telephone but it's true the amount of tax you have to pay isDON'T MAINTAIN IT!!!!</s>,<pad> Price and tax are the same as old Royal Mail payment methods.</s>,2.399293,3.889017,1.489724
2,"Summarize the following conversation. #Person1#: Hello? #Person2#: Hello? #Person1#: Can I speak to Li Hong, please? #Person2#: Speaking. #Person1#: Hi, Li Hong. This is Alice. #Person2#: Hi, Alice. How are you? #Person1#: Not bad. Li Hong, I am sorry that I can't go to see Mrs. Brown with you tomorrow morning. My mother is ill. I must take care of her. #Person2#: I'm sorry to hear that. You'd better stay at home. After all, we can visit Mrs. Brown later #Person1#: OK. Bye - bye. #Person2#: ...","<pad> #Person1#: Hi, that's Li Hong, is all right. Mrs. Brown is sick. She's afraid that getting ""much better"" will take too many people.</s>",<pad> #Person1#: My mom is ill. She cannot see me tomorrow morning. We can visit Mrs. Brown later.</s>,1.016277,1.995026,0.978750
3,"Summarize the following conversation. #Person1#: Could you help me, Sir? My flight got in 15 minutes ago. Everyone else has picked up the luggage but mine hasn't come through. #Person2#: I'm sorry, Madam, I'll go and find out if there is any more to come. Summary: </s>",<pad> The flight is in 15 minutes. It does not belong and everyone is free. Easy access to the airport is cheaper.</s>,<pad> No problem. Will you help me please?</s>,1.916374,2.723552,0.807178
4,"Summarize the following conversation. #Person1#: Mom, I just finished my paper. Can you proofread it before I hand it in? #Person2#: Sure, let's take a look. Sweetie, this is terrific. Your ideas are so original. #Person1#: Thanks. #Person2#: I can tell you worked hard on it. #Person1#: I really did! I started thinking about what I wanted to say three weeks ago. #Person2#: Well, it was definitely worth all the time. #Person1#: Let's just hope my teacher agrees. Summary: </s>",<pad> Ai Taylor's ready.</s>,<pad> Sue and her teacher will study this week.</s>,2.972778,3.645579,0.672801
5,"Summarize the following conversation. #Person1#: I would like to order some internet today. #Person2#: What kind would you like? #Person1#: What kind of internet is there? #Person2#: You can get DEL or dial-up. #Person1#: Which of those two is best? #Person2#: I would recommend DEL. #Person1#: So that one better? #Person2#: It's better because it doesn't tie up the phone. #Person1#: What do you mean by that? #Person2#: DEL isn't connected through your phone line, but dial-up is. #Person1#: S...",<pad> #Person1#: I would like to order some internet.</s>,<pad> Talk to the store clerk to know what you are getting then.</s>,2.652600,3.199640,0.547040
6,"Summarize the following conversation.